In [ ]:
import pandas as pd

# Load data training dan testing
training_df = pd.read_csv('training_dataset.csv')
testing_df = pd.read_csv('testing_dataset.csv')

# Cek head data training
print(training_df.head())

# Cek head data testing
print(testing_df.head())

   customer_number  usia               pekerjaan status_perkawinan  \
0           531036    63  sosial media specialis           menikah   
1           999241    43                 teknisi           menikah   
2           995002    29  sosial media specialis            lajang   
3           932750    40           pekerja kasar           menikah   
4           684699    40  sosial media specialis            lajang   

          pendidikan gagal_bayar_sebelumnya pinjaman_rumah pinjaman_pribadi  \
0  Pendidikan Tinggi                     no            yes               no   
1  Pendidikan Tinggi                     no            yes               no   
2  Pendidikan Tinggi                     no            yes              yes   
3                SMA                     no             no               no   
4  Pendidikan Tinggi                     no             no               no   

  jenis_kontak bulan_kontak_terakhir  ... hari_sejak_kontak_sebelumnya  \
0     cellular                

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Copy supaya ga ngerubah data asli
X_train = training_df.drop(columns=['customer_number', 'berlangganan_deposito']).copy()
y_train = training_df['berlangganan_deposito']

X_test = testing_df.drop(columns=['customer_number']).copy()

# Ganti nilai 999 jadi NaN di 'hari_sejak_kontak_sebelumnya'
X_train['hari_sejak_kontak_sebelumnya'].replace(999, pd.NA, inplace=True)
X_test['hari_sejak_kontak_sebelumnya'].replace(999, pd.NA, inplace=True)

# Buat fitur baru 'belum_dihubungi' -> 1 kalau NaN, 0 kalau ada nilai
X_train['belum_dihubungi'] = X_train['hari_sejak_kontak_sebelumnya'].isna().astype(int)
X_test['belum_dihubungi'] = X_test['hari_sejak_kontak_sebelumnya'].isna().astype(int)

# Isi NaN dengan median kolom
X_train['hari_sejak_kontak_sebelumnya'].fillna(X_train['hari_sejak_kontak_sebelumnya'].median(), inplace=True)
X_test['hari_sejak_kontak_sebelumnya'].fillna(X_test['hari_sejak_kontak_sebelumnya'].median(), inplace=True)

# Mapping manual 'pendidikan'
pendidikan_map = {
    'TIDAK SEKOLAH': 0,
    'Tidak Tamat SD': 1,
    'SD': 2,
    'SMP': 3,
    'SMA': 4,
    'Diploma': 5,
    'Pendidikan Tinggi': 6,
    'unknown': 7
}
X_train['pendidikan'] = X_train['pendidikan'].map(pendidikan_map)
X_test['pendidikan'] = X_test['pendidikan'].map(pendidikan_map)

# Kolom kategori dengan 'unknown' yang perlu diisi
kategori_unknown_cols = ['status_perkawinan', 'gagal_bayar_sebelumnya', 'pinjaman_rumah', 'pinjaman_pribadi']
for col in kategori_unknown_cols:
    X_train[col] = X_train[col].fillna('unknown').astype(str)
    X_test[col] = X_test[col].fillna('unknown').astype(str)

# List kolom kategori
categorical_cols = [
    'pekerjaan', 'status_perkawinan', 'pulau', 'jenis_kontak', 'bulan_kontak_terakhir',
    'hari_kontak_terakhir', 'hasil_kampanye_sebelumnya', 'gagal_bayar_sebelumnya',
    'pinjaman_rumah', 'pinjaman_pribadi'
]

# Label encoding kolom kategori
for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    # apply transform juga ke data test, hati2 agar gak ada kategori baru di test ya
    X_test[col] = le.transform(X_test[col])

# List kolom numerik yang perlu discale
num_cols = [
    'usia', 'jumlah_kontak_kampanye_ini', 'jumlah_kontak_sebelumnya', 'tingkat_variasi_pekerjaan',
    'indeks_harga_konsumen', 'indeks_kepercayaan_konsumen', 'suku_bunga_euribor_3bln',
    'jumlah_pekerja', 'hari_sejak_kontak_sebelumnya'
]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("Preprocessing selesai bro!")


<ipython-input-2-eeb0bafd177d>:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train['hari_sejak_kontak_sebelumnya'].replace(999, pd.NA, inplace=True)
<ipython-input-2-eeb0bafd177d>:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value,

Preprocessing selesai bro!


In [ ]:
from imblearn.over_sampling import SMOTENC
from sklearn.model_selection import train_test_split

# Tentuin indeks kolom kategori
categorical_features_idx = [X_train.columns.get_loc(col) for col in categorical_cols]

# Terapkan SMOTENC
smote_nc = SMOTENC(categorical_features=categorical_features_idx, random_state=42)
X_resampled, y_resampled = smote_nc.fit_resample(X_train, y_train)

print("Sebelum SMOTE:", y_train.value_counts())
print("Setelah SMOTE:", y_resampled.value_counts())


Sebelum SMOTE: berlangganan_deposito
0    20302
1     2614
Name: count, dtype: int64
Setelah SMOTE: berlangganan_deposito
1    20302
0    20302
Name: count, dtype: int64


In [ ]:
X_train_final, X_valid, y_train_final, y_valid = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

model = RandomForestClassifier(random_state=42)
model.fit(X_train_final, y_train_final)

y_pred = model.predict(X_valid)
y_pred_proba = model.predict_proba(X_valid)[:,1]

print(classification_report(y_valid, y_pred))
print("AUC:", roc_auc_score(y_valid, y_pred_proba))

              precision    recall  f1-score   support

           0       0.92      0.93      0.92      4049
           1       0.93      0.92      0.92      4072

    accuracy                           0.92      8121
   macro avg       0.92      0.92      0.92      8121
weighted avg       0.92      0.92      0.92      8121

AUC: 0.9724697965638027


In [ ]:
# Prediksi probabilitas kelas 1 (berlangganan deposito)
y_test_proba = model.predict_proba(X_test)[:, 1]

# Buat dataframe hasil prediksi supaya gampang dilihat / simpan
hasil_prediksi = pd.DataFrame({
    'customer_number': testing_df['customer_number'],
    'deposito_prob': y_test_proba
})

print(hasil_prediksi.head())

   customer_number  deposito_prob
0           445420           0.09
1           585604           0.04
2           888824           0.06
3           816820           0.06
4           542716           0.06


In [ ]:
# Prediksi probabilitas kelas 1 (berlangganan deposito)
y_test_proba = model.predict_proba(X_test)[:, 1]

# Buat dataframe hasil prediksi sesuai format submission
hasil_prediksi = pd.DataFrame({
    'customer_number': testing_df['customer_number'],  # dari data testing asli
    'berlangganan_deposito': y_test_proba
})

# Save ke CSV
hasil_prediksi.to_csv('submission.csv', index=False)

print(hasil_prediksi.head())

   customer_number  berlangganan_deposito
0           445420                   0.09
1           585604                   0.04
2           888824                   0.06
3           816820                   0.06
4           542716                   0.06
